# Actividad 2: Kaggle, Databricks CE, Spark y SQL

**Estudiante:** MARLON MONTERROSA, YIMY MOSQUERA, RAFAEL GONZALEZ
**Dataset:** Iris (`uciml/iris`)  
**Objetivo:** configurar y evidenciar el entorno de Databricks CE, ingerir datos desde Kaggle, persistirlos como tabla y validarlos mediante PySpark y SQL.



## 1. Arquitectura y esquema de almacenamiento

Flujo propuesto: **Kaggle API → Unity Catalog Volume → DataFrame Spark → tabla Delta**.

| Campo | Tipo Spark | Nulo | Descripción |
|---|---|---|---|
| `id` | INT | No | Identificador único |
| `sepal_length_cm` | DOUBLE | No | Longitud del sépalo en centímetros |
| `sepal_width_cm` | DOUBLE | No | Ancho del sépalo en centímetros |
| `petal_length_cm` | DOUBLE | No | Longitud del pétalo en centímetros |
| `petal_width_cm` | DOUBLE | No | Ancho del pétalo en centímetros |
| `species` | STRING | No | Especie de Iris |

La tabla no se particiona porque el dataset solo contiene 150 registros. Particionarlo produciría archivos pequeños sin beneficio.

## 2. Configuración y evidencia de la infraestructura

La siguiente celda imprime la versión del Runtime cuando está expuesta por el entorno, el tipo de clúster, núcleos, memoria máxima estimada, autoscaling y versiones de Spark/Python. Algunas propiedades pueden aparecer como `No disponible` en Databricks Community Edition.

In [0]:
import os
import platform

print("=== INFRAESTRUCTURA DATABRICKS ===")
print("Tipo de cómputo: Serverless")
print("Administración: Databricks administra núcleos, RAM y autoscaling")
print("Núcleos/RAM: no visibles para el usuario en Serverless")
print("Autoscaling: administrado automáticamente por Databricks")
print(f"Versión de Python: {platform.python_version()}")
print(f"Versión de Spark: {spark.version}")

print("\n=== VERSIÓN DEL ENTORNO ===")
display(spark.sql("SELECT current_version() AS databricks_runtime"))

=== INFRAESTRUCTURA DATABRICKS ===
Tipo de cómputo: Serverless
Administración: Databricks administra núcleos, RAM y autoscaling
Núcleos/RAM: no visibles para el usuario en Serverless
Autoscaling: administrado automáticamente por Databricks
Versión de Python: 3.11.10
Versión de Spark: 4.1.0

=== VERSIÓN DEL ENTORNO ===


databricks_runtime
"List(18.2.x-aarch64-photon-scala2.13, null, 2569d4da18c855800f780e438ea7760951a5797c, 497c462963b8f6210045853c56381870e985763e)"


### Interpretación de la infraestructura

El notebook utiliza cómputo **Serverless**. Databricks administra
automáticamente los recursos, núcleos, memoria RAM y autoscaling.

Por restricciones de Serverless, no está permitido acceder directamente a
`spark.sparkContext` ni a la JVM de Spark. Por esta razón, estas propiedades
no pueden consultarse desde el notebook.

La configuración visible es:

- Tipo de cómputo: Serverless.
- SQL Warehouse: Serverless Starter Warehouse.
- Tamaño: 2X-Small.
- Máximo de instancias: 1.
- Autoscaling: administrado automáticamente por Databricks.

**Interpretación:** esta salida evidencia el Runtime, el cómputo disponible y el autoscaling. En CE algunos datos administrativos son gestionados por Databricks y no se exponen al usuario.

In [0]:
print("=== CONFIGURACIÓN DISPONIBLE DE SPARK ===")
print(f"spark.version = {spark.version}")

configuracion = spark.sql("SET -v")
display(configuracion)

=== CONFIGURACIÓN DISPONIBLE DE SPARK ===
spark.version = 4.1.0


key,value,meaning,Since version
spark.databricks.execution.timeout,9000,Timeout in seconds for query executions that can be changed by user.,
spark.sql.ansi.enabled,true,"When true, Spark SQL uses an ANSI compliant dialect instead of being Hive compliant. For example, Spark will throw an exception at runtime instead of returning null results when the inputs to a SQL operator/function are invalid. For full details of this dialect, you can find them in the section ""ANSI Compliance"" of Spark's documentation. Some ANSI dialect features may be not from the ANSI SQL standard directly, but their behaviors align with ANSI SQL's style",3.0.0
spark.sql.files.maxPartitionBytes,128MB,"The maximum number of bytes to pack into a single partition when reading files. This configuration is effective only when using file-based sources such as Parquet, JSON and ORC.",2.0.0
spark.sql.session.timeZone,Etc/UTC,"The ID of session local timezone in the format of either region-based zone IDs or zone offsets. Region IDs must have the form 'area/city', such as 'America/Los_Angeles'. Zone offsets must be in the format '(+|-)HH', '(+|-)HH:mm' or '(+|-)HH:mm:ss', e.g '-08', '+01:00' or '-13:33:33'. Also 'UTC' and 'Z' are supported as aliases of '+00:00'. Other short names are not recommended to use because they can be ambiguous.",2.2.0
spark.sql.shuffle.partitions,auto,"The default number of partitions to use when shuffling data for joins or aggregations. Or set to 'auto' to enable Auto Optimized Shuffle, which will automatically determine this number based on the query plan and the query input data size.",1.1.0


### Estructura de almacenamiento

Se utilizará `/Volumes/workspace/actividad_2/datos` como zona de aterrizaje del CSV y una tabla Delta administrada llamada `actividad_2.iris`. La siguiente celda crea y evidencia ambas ubicaciones.

In [0]:
# Obtener catálogo actual
CATALOG_NAME = spark.sql("SELECT current_catalog()").first()[0]

# Crear esquema y Volume
spark.sql("CREATE SCHEMA IF NOT EXISTS actividad_2")
spark.sql("CREATE VOLUME IF NOT EXISTS actividad_2.datos")

# Definir rutas
VOLUME_DIR = f"/Volumes/{CATALOG_NAME}/actividad_2/datos"
CSV_PATH = f"{VOLUME_DIR}/Iris.csv"
TABLE_NAME = f"{CATALOG_NAME}.actividad_2.iris"

print("=== ESTRUCTURA DE ALMACENAMIENTO ===")
print(f"Catálogo: {CATALOG_NAME}")
print("Esquema: actividad_2")
print("Volume: datos")
print(f"Ruta del Volume: {VOLUME_DIR}")
print(f"Ruta del CSV: {CSV_PATH}")
print(f"Tabla final: {TABLE_NAME}")

display(dbutils.fs.ls(VOLUME_DIR))

=== ESTRUCTURA DE ALMACENAMIENTO ===
Catálogo: workspace
Esquema: actividad_2
Volume: datos
Ruta del Volume: /Volumes/workspace/actividad_2/datos
Ruta del CSV: /Volumes/workspace/actividad_2/datos/Iris.csv
Tabla final: workspace.actividad_2.iris


[]

## 3. Obtención del dataset desde Kaggle

Se usa la API de Kaggle. Cree un token en **Kaggle → Settings → API → Create New Token**. Copie `username` y `key` de `kaggle.json` en los widgets, pero no publique la clave ni la muestre en capturas.

In [0]:
%pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("kaggle_username", "", "Kaggle username")
dbutils.widgets.text("kaggle_key", "", "Kaggle API key")
print("Widgets creados.")

Widgets creados.


In [0]:
import os
from kaggle.api.kaggle_api_extended import KaggleApi

# Obtener credenciales de los widgets
username = dbutils.widgets.get("kaggle_username").strip()
key = dbutils.widgets.get("kaggle_key").strip()

if not username or not key:
    raise ValueError(
        "Complete los widgets Kaggle username y Kaggle API key."
    )

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

# Descargar directamente al Unity Catalog Volume
api = KaggleApi()
api.authenticate()

api.dataset_download_files(
    "uciml/iris",
    path=VOLUME_DIR,
    unzip=True,
    force=True
)

print("=== DESCARGA COMPLETADA ===")
print("Dataset descargado desde Kaggle: uciml/iris")
print(f"Ruta utilizada: {VOLUME_DIR}")
print(f"Archivo esperado: {CSV_PATH}")

display(dbutils.fs.ls(VOLUME_DIR))

Dataset URL: https://www.kaggle.com/datasets/uciml/iris
=== DESCARGA COMPLETADA ===
Dataset descargado desde Kaggle: uciml/iris
Ruta utilizada: /Volumes/workspace/actividad_2/datos
Archivo esperado: /Volumes/workspace/actividad_2/datos/Iris.csv


path,name,size,modificationTime
dbfs:/Volumes/workspace/actividad_2/datos/Iris.csv,Iris.csv,5107,1780761491000
dbfs:/Volumes/workspace/actividad_2/datos/database.sqlite,database.sqlite,10240,1780761491000


**Validación:** la salida anterior confirma el origen Kaggle, la ruta de almacenamiento y la existencia del archivo.

## 4. Lectura con esquema explícito y creación de tabla

Aplicar un esquema explícito evita que Spark interprete incorrectamente los tipos y permite validar las columnas esperadas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType
)

source_schema = StructType([
    StructField("Id", IntegerType(), False),
    StructField("SepalLengthCm", DoubleType(), False),
    StructField("SepalWidthCm", DoubleType(), False),
    StructField("PetalLengthCm", DoubleType(), False),
    StructField("PetalWidthCm", DoubleType(), False),
    StructField("Species", StringType(), False)
])

df = (
    spark.read
    .option("header", True)
    .schema(source_schema)
    .csv(CSV_PATH)
    .select(
        F.col("Id").alias("id"),
        F.col("SepalLengthCm").alias("sepal_length_cm"),
        F.col("SepalWidthCm").alias("sepal_width_cm"),
        F.col("PetalLengthCm").alias("petal_length_cm"),
        F.col("PetalWidthCm").alias("petal_width_cm"),
        F.col("Species").alias("species")
    )
)

df.printSchema()
print(f"Cantidad de registros leídos: {df.count()}")
display(df.limit(10))

root
 |-- id: integer (nullable = true)
 |-- sepal_length_cm: double (nullable = true)
 |-- sepal_width_cm: double (nullable = true)
 |-- petal_length_cm: double (nullable = true)
 |-- petal_width_cm: double (nullable = true)
 |-- species: string (nullable = true)

Cantidad de registros leídos: 150


id,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa
4,4.6,3.1,1.5,0.2,Iris-setosa
5,5.0,3.6,1.4,0.2,Iris-setosa
6,5.4,3.9,1.7,0.4,Iris-setosa
7,4.6,3.4,1.4,0.3,Iris-setosa
8,5.0,3.4,1.5,0.2,Iris-setosa
9,4.4,2.9,1.4,0.2,Iris-setosa
10,4.9,3.1,1.5,0.1,Iris-setosa


**Validación:** `printSchema()` confirma los nombres y tipos; el conteo comprueba que se cargaron 150 filas; la muestra permite inspeccionar visualmente los valores.

In [0]:
nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])
duplicate_ids = df.groupBy("id").count().filter("count > 1").count()
print(f"IDs duplicados: {duplicate_ids}")
display(nulls)

assert df.count() == 150, "El número de filas no coincide con lo esperado."
assert duplicate_ids == 0, "Existen IDs duplicados."
assert sum(nulls.first()) == 0, "Existen valores nulos."

spark.sql("CREATE DATABASE IF NOT EXISTS actividad_2")
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)
print(f"Tabla Delta creada correctamente: {TABLE_NAME}")

IDs duplicados: 0


id,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
0,0,0,0,0,0


Tabla Delta creada correctamente: workspace.actividad_2.iris


**Validación:** los controles verifican integridad básica: 150 filas, IDs únicos y ausencia de nulos. La confirmación final evidencia la persistencia como tabla Delta.

## 5. Validaciones equivalentes en PySpark y SQL

In [0]:
table_df = spark.table(TABLE_NAME)
print("=== ESQUEMA DE LA TABLA ===")
table_df.printSchema()
print("=== DESCRIPCIÓN ESTADÍSTICA ===")
table_df.describe().show(truncate=False)
print("=== CONTEO Y FILTRO: pétalo mayor a 5 cm ===")
print(f"COUNT(*): {table_df.count()}")
table_df.filter(F.col("petal_length_cm") > 5).show(10, truncate=False)

=== ESQUEMA DE LA TABLA ===
root
 |-- id: integer (nullable = true)
 |-- sepal_length_cm: double (nullable = true)
 |-- sepal_width_cm: double (nullable = true)
 |-- petal_length_cm: double (nullable = true)
 |-- petal_width_cm: double (nullable = true)
 |-- species: string (nullable = true)

=== DESCRIPCIÓN ESTADÍSTICA ===
+-------+------------------+------------------+------------------+------------------+------------------+--------------+
|summary|id                |sepal_length_cm   |sepal_width_cm    |petal_length_cm   |petal_width_cm    |species       |
+-------+------------------+------------------+------------------+------------------+------------------+--------------+
|count  |150               |150               |150               |150               |150               |150           |
|mean   |75.5              |5.843333333333335 |3.0540000000000007|3.7586666666666693|1.1986666666666672|NULL          |
|stddev |43.445367992456916|0.8280661279778629|0.4335943113621737|1.764420

**Propósito:** comprobar metadatos, rangos estadísticos, total de filas y funcionamiento de filtros mediante PySpark.

In [0]:
spark_grouped = (table_df.groupBy("species")
    .agg(F.count("*").alias("cantidad"),
         F.round(F.avg("sepal_length_cm"), 2).alias("promedio_sepalo_cm"),
         F.round(F.avg("petal_length_cm"), 2).alias("promedio_petalo_cm"))
    .orderBy("species"))
display(spark_grouped)

species,cantidad,promedio_sepalo_cm,promedio_petalo_cm
Iris-setosa,50,5.01,1.46
Iris-versicolor,50,5.94,4.26
Iris-virginica,50,6.59,5.55


**Propósito:** el `GROUP BY` de PySpark valida que existen tres especies con 50 registros cada una y calcula promedios coherentes.

In [0]:
%sql
DESCRIBE TABLE actividad_2.iris;

col_name,data_type,comment
id,int,null
sepal_length_cm,double,null
sepal_width_cm,double,null
petal_length_cm,double,null
petal_width_cm,double,null
species,string,null


In [0]:
%sql
SHOW CREATE TABLE actividad_2.iris;

createtab_stmt
"CREATE TABLE workspace.actividad_2.iris ( id INT, sepal_length_cm DOUBLE, sepal_width_cm DOUBLE, petal_length_cm DOUBLE, petal_width_cm DOUBLE, species STRING COLLATE UTF8_BINARY) USING delta TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM actividad_2.iris;

total_registros
150


In [0]:
%sql
SELECT *
FROM actividad_2.iris
WHERE petal_length_cm > 5
ORDER BY id
LIMIT 10;

id,sepal_length_cm,sepal_width_cm,petal_length_cm,petal_width_cm,species
84,6.0,2.7,5.1,1.6,Iris-versicolor
101,6.3,3.3,6.0,2.5,Iris-virginica
102,5.8,2.7,5.1,1.9,Iris-virginica
103,7.1,3.0,5.9,2.1,Iris-virginica
104,6.3,2.9,5.6,1.8,Iris-virginica
105,6.5,3.0,5.8,2.2,Iris-virginica
106,7.6,3.0,6.6,2.1,Iris-virginica
108,7.3,2.9,6.3,1.8,Iris-virginica
109,6.7,2.5,5.8,1.8,Iris-virginica
110,7.2,3.6,6.1,2.5,Iris-virginica


In [0]:
%sql
SELECT species,
       COUNT(*) AS cantidad,
       ROUND(AVG(sepal_length_cm), 2) AS promedio_sepalo_cm,
       ROUND(AVG(petal_length_cm), 2) AS promedio_petalo_cm
FROM actividad_2.iris
GROUP BY species
ORDER BY species;

species,cantidad,promedio_sepalo_cm,promedio_petalo_cm
Iris-setosa,50,5.01,1.46
Iris-versicolor,50,5.94,4.26
Iris-virginica,50,6.59,5.55


**Interpretación SQL:** `DESCRIBE TABLE` valida columnas y tipos; `SHOW CREATE TABLE` evidencia formato y definición; `COUNT(*)` confirma 150 registros; `LIMIT` y el filtro permiten inspección; el `GROUP BY` debe coincidir con PySpark, demostrando consistencia entre ambas interfaces.

## 6. Comparación: SQL frente a Spark/PySpark

| Aspecto | SQL | Spark / PySpark |
|---|---|---|
| Facilidad | Sintaxis declarativa, breve y familiar | Requiere conocer Python y la API de Spark |
| Expresividad | Excelente para SELECT, JOIN y GROUP BY | Adecuado para pipelines y lógica compleja |
| Integración | Fácil integración con BI y analistas | Integra DataFrames, RDD, UDFs y MLlib |
| Escalabilidad | Spark SQL escala usando el motor Spark | Diseñado para procesamiento distribuido |
| Limitaciones | Menos flexible para lógica procedural y UDFs complejas | Mayor curva de aprendizaje y ajustes de rendimiento |

**Conclusión:** SQL y PySpark se complementan. SQL simplifica el análisis tabular y la integración con BI; PySpark facilita la ingesta, las validaciones reutilizables y los pipelines complejos. En DataFrames, ambos aprovechan el optimizador de Spark.

## 7. Conclusiones y entrega

Se configuró y evidenció correctamente el entorno Databricks Community Edition utilizando cómputo Serverless. Debido a que este entorno tiene deshabilitado el DBFS público, el dataset Iris obtenido desde Kaggle se almacenó en un Unity Catalog Volume. Posteriormente, los datos fueron cargados mediante Spark y persistidos en la tabla Delta workspace.actividad_2.iris.

PySpark permitió definir un esquema explícito, comprobar la cantidad de registros, validar valores nulos, detectar identificadores duplicados y crear la tabla Delta. Por su parte, SQL facilitó la consulta de metadatos, la exploración de registros y la realización de conteos, filtros y agrupaciones.

Las validaciones realizadas con PySpark y SQL produjeron resultados consistentes: se identificaron 150 registros distribuidos entre tres especies, con 50 observaciones para cada una. Esto demuestra que ambas herramientas pueden utilizarse de forma complementaria para desarrollar procesos confiables de almacenamiento, procesamiento y análisis de datos.

In [0]:
os.environ.pop("KAGGLE_USERNAME", None)
os.environ.pop("KAGGLE_KEY", None)
dbutils.widgets.removeAll()
print("Credenciales eliminadas de las variables de entorno de la sesión.")

Credenciales eliminadas de las variables de entorno de la sesión.
